# Banker's Wrapped — Demo Runbook

Interactive walkthrough of the full pipeline against multiple synthetic datasets.

**Prerequisites**
- Backend running: `make dev` or `make demo-start`
- `.env` configured with `GMI_API_KEY`, `NVIDIA_NIM_API_KEY`, `B2_*`
- `pip install httpx pandas matplotlib ipython` (if running outside the project venv)

---

## Scenarios

| # | Dataset | Expected Personality | Focus |
| --- | --- | --- | --- |
| 1 | Smoke test | — | Verify backend is live |
| 2 | `transactions_jan_2026.csv` | Financial Builder | High income, moderate savings |
| 3 | `transactions_q4_2025.csv` | Financial Explorer | Higher spend diversity |
| 4 | Side-by-side comparison | Both | Metrics visualised |
| 5 | B2 artifact inspection | — | Show everything stored per session |

In [ ]:
import json
import time
from pathlib import Path

import httpx

BASE_URL = "http://127.0.0.1:8000"
SYNTHETIC = Path("../data/synthetic")

print(f"Backend: {BASE_URL}")
print(f"Synthetic data: {SYNTHETIC.resolve()}")

---
## Scenario 1 — Smoke Test

Verify the backend is reachable before running any expensive pipeline calls.

In [ ]:
r = httpx.get(f"{BASE_URL}/api/v1/health")
data = r.json()
assert r.status_code == 200, f"Health check failed: {r.status_code}"
print(f"Status  : {data['status']}")
print(f"Version : {data.get('version', 'n/a')}")
print("Backend is ready.")

---
## Scenario 2 — Financial Builder (January 2026)

22 transactions · high income · moderate saver.  
Expected personality: **Financial Builder**.

In [ ]:
def run_pipeline(csv_path: Path, label: str) -> dict:
    """POST a CSV to the pipeline and return the parsed JSON response."""
    print(f"Running pipeline for {label} ({csv_path.name}) ...")
    t0 = time.time()
    with open(csv_path, "rb") as f:
        r = httpx.post(
            f"{BASE_URL}/api/v1/recap/generate",
            files={"file": (csv_path.name, f, "text/csv")},
            timeout=900,
        )
    elapsed = round(time.time() - t0, 1)
    r.raise_for_status()
    result = r.json()
    result["_elapsed_s"] = elapsed
    return result


def print_summary(result: dict) -> None:
    ins = result["insights"]
    print(f"  Session      : {result['session_id']}")
    print(f"  Wall clock   : {result['_elapsed_s']} s  (pipeline: {result['processing_time_ms']} ms)")
    print(f"  Personality  : {ins['personality']}")
    print(f"  Period       : {ins['period_label']}")
    print(f"  Income       : {ins['currency']} {ins['total_income']:>12,.2f}")
    print(f"  Expenses     : {ins['currency']} {ins['total_expenses']:>12,.2f}")
    print(f"  Savings rate : {ins['savings_rate']:.1f}%")
    top = ins["top_categories"][:5]
    cats = ", ".join(f"{c['category']} ({c['percentage']:.0f}%)" for c in top)
    print(f"  Top spend    : {cats}")
    print(f"  Video URL    : {result['video_url'][:80]}...")


jan = run_pipeline(SYNTHETIC / "transactions_jan_2026.csv", "Financial Builder")
print_summary(jan)

### View the generated video inline

In [ ]:
from IPython.display import Video, display

video_url = jan["video_url"]
print(f"Video URL: {video_url}")
display(Video(video_url, width=800, embed=False))

---
## Scenario 3 — Financial Explorer (Q4 2025)

39 transactions · diverse categories · higher lifestyle spend.  
Expected personality: **Financial Explorer**.

In [ ]:
q4 = run_pipeline(SYNTHETIC / "transactions_q4_2025.csv", "Financial Explorer")
print_summary(q4)

In [ ]:
display(Video(q4["video_url"], width=800, embed=False))

---
## Scenario 4 — Side-by-Side Comparison

Visualise key metrics across both datasets to show how the pipeline adapts its narrative to each user's financial fingerprint.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

datasets = [
    ("Jan 2026\n(Builder)",  jan["insights"]),
    ("Q4 2025\n(Explorer)", q4["insights"]),
]

labels   = [d[0] for d in datasets]
incomes  = [d[1]["total_income"]   for d in datasets]
expenses = [d[1]["total_expenses"] for d in datasets]
savings  = [d[1]["savings_rate"]   for d in datasets]

x = np.arange(len(labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Banker's Wrapped — Pipeline Output Comparison", fontsize=13, fontweight="bold")

# ── Income vs Expenses ────────────────────────────────────────────────────────
ax = axes[0]
bars_inc = ax.bar(x - width/2, incomes,  width, label="Income",   color="#4A90E2")
bars_exp = ax.bar(x + width/2, expenses, width, label="Expenses", color="#E8392A")
ax.set_title("Income vs Expenses")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.legend()
ax.bar_label(bars_inc, fmt="$%.0f", padding=3, fontsize=8)
ax.bar_label(bars_exp, fmt="$%.0f", padding=3, fontsize=8)

# ── Top Spend Categories (Jan 2026) ──────────────────────────────────────────
ax2 = axes[1]
top_cats = jan["insights"]["top_categories"][:5]
cat_names = [c["category"] for c in top_cats]
cat_pcts  = [c["percentage"] for c in top_cats]
colors = ["#4A90E2", "#E8392A", "#76B900", "#FF9500", "#9B59B6"]
wedges, texts, autotexts = ax2.pie(
    cat_pcts, labels=cat_names, autopct="%1.0f%%",
    colors=colors, startangle=140, textprops={"fontsize": 8},
)
ax2.set_title("Jan 2026 — Top Spend Categories")

plt.tight_layout()
plt.savefig("../logs/demo_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to logs/demo_comparison.png")

### Pipeline timing breakdown

In [ ]:
# Measured timings from a live run (2026-06-26)
stages = [
    ("CSV parse",           0.005),
    ("Analytics",           0.001),
    ("NIM narrative",      31.0),
    ("Images × 4 parallel", 155.0),
    ("FFmpeg compose",       1.0),
    ("B2 upload",            2.0),
]

names = [s[0] for s in stages]
times = [s[1] for s in stages]
total = sum(times)

fig, ax = plt.subplots(figsize=(10, 3))
bar_colors = ["#009688", "#009688", "#76B900", "#0066CC", "#007808", "#E8392A"]
bars = ax.barh(names, times, color=bar_colors, edgecolor="white")
ax.set_xlabel("Seconds")
ax.set_title(f"Pipeline Stage Timing  (total: ~{total:.0f} s ≈ {total/60:.1f} min)")
ax.bar_label(bars, fmt="%.1f s", padding=4, fontsize=9)
ax.set_xlim(0, max(times) * 1.15)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../logs/pipeline_timing.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Total wall-clock: {total:.1f} s  |  Image generation dominates ({100*times[3]/total:.0f}% of total)")

---
## Scenario 5 — B2 Artifact Inspection

Show the full structured artifact layout that was stored for the January pipeline run.  
Every session leaves a complete, traceable trail in Backblaze B2.

In [ ]:
b2_keys = jan["b2_keys"]
session_id = jan["session_id"]

print(f"Session: {session_id}")
print()
print("B2 artifact layout")
print("==================")
for key_name, b2_key in sorted(b2_keys.items()):
    size_hint = "" 
    print(f"  {key_name:<12}  {b2_key}")

print()
print("session_metadata.json (provenance trail)")
print("========================================")
meta = {
    "session_id":        session_id,
    "personality":       jan["insights"]["personality"],
    "pipeline_version":  jan.get("pipeline_version", "1.1.0"),
    "models_used":       jan.get("models_used", {}),
    "processing_time_ms": jan["processing_time_ms"],
    "video_url":         jan["video_url"],
}
print(json.dumps(meta, indent=2))

---
## Summary

| Scenario | Status | Time |
| --- | --- | --- |
| Smoke test | Ready | <1 s |
| Jan 2026 — Financial Builder | Video in B2 | ~195 s |
| Q4 2025 — Financial Explorer | Video in B2 | ~195 s |
| Metrics chart | Rendered | — |
| B2 artifact trail | Inspected | — |

**Key takeaway:** upload a CSV, receive a personalised 60-second financial recap video in under 4 minutes — fully stored, traceable, and reproducible on Backblaze B2.